# Designing the reward programme

This notebook costs the illustrative tier schedule for the 250 members of `test.csv`, shows where the uncertainty in that cost comes from, and searches for thresholds that reach more members at no more risk. It follows `docs/methodology.md`, section 9.

In [1]:
import warnings

import numpy as np
import pandas as pd

from loyalty_spend.business.noise import compare_noise_models
from loyalty_spend.business.programme import design_programme
from loyalty_spend.business.rewards import MarginalSchedule, RewardSchedule
from loyalty_spend.config import load_config
from loyalty_spend.data.io import load_clean

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
cfg = load_config("../configs/default.yaml")
train, _ = load_clean("../data/raw/train.csv", impute=False)
test, _ = load_clean("../data/raw/test.csv", impute=False, require_target=False)
cfg.rewards.tiers

((0.0, 0.0), (100.0, 0.02), (125.0, 0.03), (140.0, 0.05))

## Whole-spend and marginal schedules

Under a whole-spend schedule a member's whole annual spend earns the rate of the highest tier reached, so the reward jumps at every threshold. Under a marginal schedule each rate applies only to the spend inside its band, like income tax brackets, and the reward is continuous.

In [2]:
schedule = RewardSchedule(cfg.rewards.tiers)
marginal = MarginalSchedule(cfg.rewards.tiers)
spend = np.array([99.0, 100.0, 124.0, 125.0, 141.0])
pd.DataFrame({"spend": spend, "whole spend": schedule.reward(spend), "marginal": marginal.reward(spend)})

,spend,whole spend,marginal
0,99.0,0.00,0.00
1,100.0,2.00,0.00
2,124.0,2.48,0.48
3,125.0,3.75,0.50
4,141.0,7.05,1.00


## Cost and risk for test.csv

Spend is drawn from a Gaussian model whose variance is fitted to `train.csv`. All schedules are costed on the same draws, so their differences carry little Monte Carlo noise.

In [3]:
design = design_programme(train, test, cfg.rewards.tiers, n_simulations=cfg.rewards.n_simulations, seed=cfg.rewards.seed)
design.schedules

,schedule,expected_cost,cost_sd,p05,p95,difference,difference_se
0,whole spend,1218.423,3.038,1212.583,1223.473,0.000,0.000
1,marginal,218.585,0.202,218.254,218.915,-999.838,0.042


In [4]:
pd.Series(design.tail_risk)

level                         0.950
var                        1223.473
expected_shortfall         1224.356
var_over_expected             5.050
shortfall_over_expected       5.933
dtype: float64

The expected cost is a little under 4% of expected spend, and the 95% expected shortfall is about five units above the mean: the cost is predictable because most members are far from a threshold.

## What moves the cost

Central differences on common random numbers give the derivative of expected cost with respect to each threshold and rate. For a whole-spend schedule, raising a threshold moves the members just below it into a lower tier.

In [5]:
design.sensitivity

,tier,threshold,rate,d_cost_d_rate,d_cost_d_threshold,d_cost_d_threshold_se
0,0,0.0,0.00,3540.614,NaN,NaN
1,1,100.0,0.02,1211.437,0.000,0.000
2,2,125.0,0.03,8814.171,-0.971,0.006
3,3,140.0,0.05,18595.380,-22.592,0.028


Raising the top threshold by one unit lowers the expected cost by about 23; the other thresholds matter far less because few members are near them. A whole-spend schedule also gives a member just below a threshold a reason to spend the difference. If members within a window below a threshold step up with some probability, the whole-spend schedule costs about 60 more (5%), while the marginal schedule, which pays nothing extra for crossing, is unaffected.

In [6]:
design.behaviour

,schedule,expected_cost_static,expected_cost_with_response,cost_increase,extra_spend
0,whole spend,1218.423,1278.390,59.967,49.53
1,marginal,218.585,218.585,0.000,0.00


## Searching thresholds under a budget

Threshold shifts on a grid are costed on shared draws. A schedule is feasible if the 95th percentile of its cost stays within the budget, here the 95th percentile of the current schedule; feasible schedules are ranked by reach, the share of members in a paid tier.

In [7]:
design.search.head(8)

,thresholds,expected_cost,cost_quantile,reach,feasible
0,"90,135,150",917.538,921.414,0.808,True
1,"95,135,150",917.538,921.414,0.808,True
2,"100,135,150",917.538,921.414,0.808,True
3,"105,135,150",917.538,921.414,0.808,True
4,"110,135,150",917.538,921.414,0.808,True
5,"90,130,150",944.673,948.283,0.808,True
6,"95,130,150",944.673,948.283,0.808,True
7,"100,130,150",944.673,948.283,0.808,True


Moving the top threshold from 140 to 150 and the middle one to 135 lowers the expected cost from about 1,218 to 918, by a quarter, while the same 81% of members stay in a paid tier: the top rate applies to the whole spend of every member above the top threshold, so it is where the cost is.

## How the noise model changes the answer

Forecast errors are larger for members who joined in some months than in others. The same point forecasts give slightly different cost distributions under a residual bootstrap, a Gaussian variance function and residuals scaled by joining month.

In [8]:
table, _ = compare_noise_models(train, test, schedule, cfg.rewards.n_simulations, cfg.rewards.seed)
table

,noise model,expected_total,p05,p95,expected_shortfall_95,members_near_threshold
0,residual bootstrap,1219.692,1215.289,1223.878,1224.877,2
1,"Gaussian, fitted variance",1218.423,1212.583,1223.473,1224.356,3
2,scaled residuals by month,1220.439,1215.559,1224.213,1225.776,2


## Conclusion

The expected reward cost for `test.csv` is about 1,220 with a 90% interval of roughly plus or minus four, whichever noise model is used. With the same rates a marginal schedule costs 219 instead of 1,218 and gives no reason to bunch above a threshold; matching the generosity of the whole-spend schedule would need higher rates. Within the whole-spend design, the top threshold is the lever: moving it to 150 cuts the cost by a quarter without moving anyone out of the paid tiers.